In [1]:
# This notebook will produce a feature table with mapped MS2 spectra and export the selected MS2 spectra in MSP format. 
# This will allow your data to be analyzed using GNPS or other MS2-based tools.

In [2]:
# Library imports for data processing

import pymzml
import os
import tqdm
import ftplib
import os
import intervaltree
import matchms
import pandas as pd

In [3]:
# block for downloading the data

FTP_URL  = "massive-ftp.ucsd.edu"
FTP_PATH = "/v04/MSV000090156/peak/mzml/POS_MSMS/Lab_2/"
DATA_DIR = "./GNPS_testing/"

def download_dir(ftp, remote_dir, local_dir):
    os.makedirs(local_dir, exist_ok=False)
    ftp.cwd(remote_dir)

    for name in ftp.nlst():
        try:
            # try to CWD → it's a directory
            ftp.cwd(name)
            ftp.cwd("..")
            download_dir(ftp, remote_dir + "/" + name, local_dir + "/" + name)
        except ftplib.error_perm:
            # it's a file
            local_fp = os.path.join(local_dir, name)
            with open(local_fp, "wb") as f:
                ftp.retrbinary(f"RETR {name}", f.write)
                print("✔", local_fp)


with ftplib.FTP(FTP_URL) as ftp:
    ftp.login()  # anonymous OK
    download_dir(ftp, FTP_PATH, DATA_DIR)



✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep1.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep2.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep3.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep1.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep2.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep3.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep1.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep2.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep3.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_M_Pos_MS2_Rep1.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_M_Pos_MS2_Rep2.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_M_Pos_MS2_Rep3.mzML
✔ ./GNPS_testing/Interlab-LC-MS_Lab2_PPL_Pos_MS2_Rep1.mzML


In [4]:
# Run asari for MS1 feature extraction

os.system(f"/Users/mitchjo/Library/Python/3.11/bin/asari process -i {DATA_DIR} -o {DATA_DIR} -m pos")



~~~~~~~ Hello from Asari (1.13.1) ~~~~~~~~~

Working on ~~ ./GNPS_testing/ ~~ 


Extracted Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep1.mzML to 10887 mass tracks.
Extracted Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep3.mzML to 11463 mass tracks.
Extracted Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep2.mzML to 11853 mass tracks.
Extracted Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep1.mzML to 12390 mass tracks.
Extracted Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep1.mzML to 11002 mass tracks.
Extracted Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep2.mzML to 12411 mass tracks.
Extracted Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep3.mzML to 12284 mass tracks.
Extracted Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep2.mzML to 11555 mass tracks.
Extracted Interlab-LC-MS_Lab2_M_Pos_MS2_Rep1.mzML to 11583 mass tracks.
Extracted Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep3.mzML to 11743 mass tracks.
Extracted Interlab-LC-MS_Lab2_M_Pos_MS2_Rep2.mzML to 10928 mass tracks.
Extracted Interlab-LC-MS_Lab2_M_Pos_MS2_Rep3.mzML to 11520 mass tracks.
Extracted Interlab-LC-MS_Lab2

0

In [5]:
# Set paths for dataset and find all MS2 spectra

all_mzml_files = []
for f in os.listdir(DATA_DIR):
    if f.lower().endswith("mzml"):
        all_mzml_files.append(os.path.join(DATA_DIR, f))

spectra = []
for mzml in tqdm.tqdm(all_mzml_files):
    for spec in pymzml.run.Reader(mzml):
        if spec.ms_level == 2:
            spec_rtime = spec.scan_time_in_minutes()*60
            for precursor in spec.selected_precursors:
                entry = {"spectrum": spec,
                         "rtime": spec_rtime,
                         "sample_origin": os.path.basename(mzml).rstrip(".mzML"),
                         "intensity_sum": sum(spec.i)}
                entry.update(precursor)
                spectra.append(entry)

print(f"Extracted {len(spectra)} MS2 Spectra")

100%|██████████| 13/13 [00:17<00:00,  1.37s/it]

Extracted 49500 MS2 Spectra


In [6]:
# find and read the feature table from the Asari run
for x in sorted(os.listdir(".")):
    if 'asari' in x:
        asari_dir = x
        break

PREF_FEATURE_TABLE_PATH = os.path.join(os.path.abspath("."), asari_dir, "preferred_Feature_table.tsv")
FULL_FEATURE_TABLE_PATH = os.path.join(os.path.abspath("."), asari_dir, "export/full_Feature_table.tsv")

print(f"Found preferred table at: {PREF_FEATURE_TABLE_PATH}")
print(f"Found full table at: {FULL_FEATURE_TABLE_PATH}")


Found preferred table at: /Users/mitchjo/asari_pcpfm_tutorials/tutorial/part_4/GNPS_testing_asari_project_111893828/preferred_Feature_table.tsv
Found full table at: /Users/mitchjo/asari_pcpfm_tutorials/tutorial/part_4/GNPS_testing_asari_project_111893828/export/full_Feature_table.tsv


In [ ]:
# this associated MS2 spectra to features
# the mapping is based on ppm and rt tolerance, then the most intense spectrum is selected of all matches. 
# this will output new tables co-located with the input tables and mgf files for upload to GNPS

def map_features_to_ms2(features, ms2_spectra, ppm_tol=5.0, rt_tol=10.0):
    spectrum_mz = intervaltree.IntervalTree()
    spectrum_rt = intervaltree.IntervalTree()
    id_to_spectrum = {}

    for s in ms2_spectra:
        mz_err = s['mz'] / 1e6 * ppm_tol
        key = (s['sample_origin'], s['spectrum'].ID)
        id_to_spectrum[key] = s
        spectrum_mz.addi(s['mz'] - mz_err, s['mz'] + mz_err, key)
        spectrum_rt.addi(s['rtime'] - rt_tol, s['rtime'] + rt_tol, key)

    out = []
    for f in features:
        mz_hits = {x.data for x in spectrum_mz.at(f['mz'])}
        rt_hits = {x.data for x in spectrum_rt.at(f['rtime'])}
        f['matches'] = mz_hits & rt_hits if mz_hits and rt_hits else set()
        out.append(f)

    return out, id_to_spectrum


def process_feature_table(
    feature_table_path,
    ms2_spectra,
):
    ft = pd.read_csv(feature_table_path, sep="\t")
    features = list(ft.to_dict(orient='records'))
    sample_cols = ft.columns[11:]
    non_sample_cols = ft.columns[:11]

    mapped, id_to_spectrum = map_features_to_ms2(features, ms2_spectra)

    features_w_ms2 = []
    spectra_to_export = []

    id_number_to_scan = {}

    for f in mapped:
        if not f['matches']:
            f['selected_ms2'] = ''
            features_w_ms2.append(f)
            continue

        possibles = [key for key in f['matches'] if f[key[0]] > 0]

        if possibles:
            scored = []
            for (sample_origin, spec_id) in possibles:
                spec = id_to_spectrum[(sample_origin, spec_id)]
                rt_err = abs(spec['rtime'] - f['rtime'])
                mz_err = abs(spec['mz'] - f['mz'])
                scored.append((rt_err, mz_err, spec,
                               round(spec['mz'], 4),
                               round(spec['rtime'], 4),
                               sample_origin))
            sel = sorted(scored, key=lambda x: x[1])[0]
            f['selected_ms2'] = f"{sel[3]}_{sel[4]}_{sel[5]}"
            try:
                sp = sel[2]['spectrum']
                spec_out = matchms.Spectrum(
                    sp.mz,
                    sp.i,
                    metadata={
                        "TITLE": f['id_number'][1:],
                        "PEPMASS": "0.0",
                        "CHARGE": "1",
                        "SCANS": f"{len(spectra_to_export) + 1}",
                        "COLLISION_ENERGY": "0.0",
                    }
                )
                spec_out = matchms.filtering.default_filters(spec_out)
                spec_out = matchms.filtering.normalize_intensities(spec_out)
                spectra_to_export.append(sp)
                id_number_to_scan[f['id_number']] = len(spectra_to_export)
            except:
                pass
        else:
            f['selected_ms2'] = ''
        del f['matches']
        features_w_ms2.append(f)

    out_msp_path = feature_table_path.replace(".tsv", "_ms2_spectra.mgf")
    with open(out_msp_path, "w") as fh:
        for i, sp in enumerate(spectra_to_export, start=1):
            fh.write("BEGIN IONS\n")
            fh.write(f"SCANS={i}\n")
            fh.write("PEPMASS=0.0\n")
            fh.write("CHARGE=1\n")
            fh.write("COLLISION_ENERGY=0.0\n")
            for m, inten in zip(sp.mz, sp.i):
                fh.write(f"{m} {inten}\n")
            fh.write("END IONS\n\n")

    print(f"Wrote {out_msp_path} with {i} MS2 Spectra")

    df = pd.DataFrame(features_w_ms2)
    new_df = pd.DataFrame()
    for x in non_sample_cols:
        new_df[x] = df[x]
    new_df['selected_ms2'] = df['selected_ms2']
    for x in sample_cols:
        new_df[x] = df[x]
    out_feature_table_path = feature_table_path.replace(".tsv", "_w_MS2.tsv")
    new_df.to_csv(out_feature_table_path, sep="\t", index=False)

    for_GNPS = pd.DataFrame()
    for_GNPS["row ID"]  = [id_number_to_scan.get(x, 0) for x in new_df["id_number"]]
    for_GNPS["row m/z"] = new_df['mz']
    for_GNPS['row retention time'] = new_df['rtime']
    for z in new_df.columns[12:]:
        for_GNPS[z + ' Peak area'] = new_df[z]
    for_GNPS = for_GNPS[for_GNPS["row ID"] != 0]
    for_GNPS.to_csv(feature_table_path.replace(".tsv", "_for_GNPS.csv"), index=False)
    print(f"Wrote {feature_table_path.replace(".tsv", "_for_GNPS.csv")} with {for_GNPS.shape[0]} Features")
    print(f"{round(i/for_GNPS.shape[0] * 100, 2)} percent of features have MS2")
    return for_GNPS


In [8]:
# process the preferred table into GNPS
process_feature_table(PREF_FEATURE_TABLE_PATH, spectra)

# process the full table into GNPS
process_feature_table(FULL_FEATURE_TABLE_PATH, spectra)

Wrote /Users/mitchjo/asari_pcpfm_tutorials/tutorial/part_4/GNPS_testing_asari_project_111893828/preferred_Feature_table_ms2_spectra.mgf with 1060 MS2 Spectra
Wrote /Users/mitchjo/asari_pcpfm_tutorials/tutorial/part_4/GNPS_testing_asari_project_111893828/preferred_Feature_table_for_GNPS.csv with 1060 Features
Wrote /Users/mitchjo/asari_pcpfm_tutorials/tutorial/part_4/GNPS_testing_asari_project_111893828/export/full_Feature_table_ms2_spectra.mgf with 2080 MS2 Spectra
Wrote /Users/mitchjo/asari_pcpfm_tutorials/tutorial/part_4/GNPS_testing_asari_project_111893828/export/full_Feature_table_for_GNPS.csv with 2080 Features


,row ID,row m/z,row retention time,Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep1 Peak area,Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep1 Peak area,Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep2 Peak area,Interlab-LC-MS_Lab2_A15M_Pos_MS2_Rep3 Peak area,Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep2 Peak area,Interlab-LC-MS_Lab2_A45M_Pos_MS2_Rep3 Peak area,Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep1 Peak area,Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep2 Peak area,Interlab-LC-MS_Lab2_A5M_Pos_MS2_Rep3 Peak area,Interlab-LC-MS_Lab2_M_Pos_MS2_Rep1 Peak area,Interlab-LC-MS_Lab2_M_Pos_MS2_Rep2 Peak area,Interlab-LC-MS_Lab2_M_Pos_MS2_Rep3 Peak area,Interlab-LC-MS_Lab2_PPL_Pos_MS2_Rep1 Peak area
87,1,150.1280,145.17,9169909,8022122,8603752,8199831,8967185,9490840,8286431,8244372,8528536,8201686,7312949,8164123,23700887
104,2,219.1748,562.85,15932849,9742241,11388463,12489916,16472579,15956249,9305137,13509977,13013509,12222758,4905253,14209498,1167764
109,3,151.0354,12.66,5158691,6402547,5347462,5671698,5899659,5559686,5665795,5681048,5189168,4791242,12978428,5635021,65867207
110,4,151.0354,39.64,136786991,137624031,135350369,136766574,147259082,162889651,151033922,158761627,146359705,177629063,156725256,150348593,5193203
111,5,151.0354,837.18,3864506,0,4692696,4015274,5786710,4518559,4614277,5306348,6002552,3567753,12804050,4624109,4705597
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44861,2076,799.6495,785.95,2907555,0,1468590,1609303,1814336,1704985,0,855873,1726585,5698308,0,1328507,7711190
44862,2077,799.6495,796.94,2150453,0,1863456,1547167,807622,769665,0,1006087,1750998,5465535,0,1888046,6293580
44863,2078,799.6495,814.06,1990150,0,1527309,1449927,1562780,1351872,0,723755,2737169,5130608,0,1405090,8326260
45046,2079,804.4895,578.34,9700015,0,10474733,11396014,11185451,9788961,0,11124628,9333826,9068362,0,10948242,16151888
